<a href="https://colab.research.google.com/github/shaikomerfarooq/Hitchin-Railway-Flyover-BIM/blob/main/UAE_BIM_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define your root project path on Google Drive
PROJECT_ROOT = '/content/drive/MyDrive/UAE_BIM_Projects/UAE_RES_VILLA_G1'
os.makedirs(PROJECT_ROOT, exist_ok=True)
print(f"Project Root established at: {PROJECT_ROOT}")

Mounted at /content/drive
Project Root established at: /content/drive/MyDrive/UAE_BIM_Projects/UAE_RES_VILLA_G1


In [2]:
# ISO 19650 CDE Folders
iso_containers = [
    "01_WIP",
    "02_SHARED",
    "03_PUBLISHED",
    "04_ARCHIVE"
]

for container in iso_containers:
    container_path = os.path.join(PROJECT_ROOT, container)
    os.makedirs(container_path, exist_ok=True)
    print(f"Created Container: {container_path}")

print("\nISO 19650 CDE structure successfully built in Google Drive.")

Created Container: /content/drive/MyDrive/UAE_BIM_Projects/UAE_RES_VILLA_G1/01_WIP
Created Container: /content/drive/MyDrive/UAE_BIM_Projects/UAE_RES_VILLA_G1/02_SHARED
Created Container: /content/drive/MyDrive/UAE_BIM_Projects/UAE_RES_VILLA_G1/03_PUBLISHED
Created Container: /content/drive/MyDrive/UAE_BIM_Projects/UAE_RES_VILLA_G1/04_ARCHIVE

ISO 19650 CDE structure successfully built in Google Drive.


In [3]:
import json
from datetime import datetime

cde_manifest = {
    "Project_Name": "UAE-RES-VILLA-G1",
    "BIM_Standard": "ISO 19650-2",
    "CDE_Root": PROJECT_ROOT,
    "Containers": {
        "01_WIP": "Work in progress - internal engineering drafts",
        "02_SHARED": "Shared container - coordinate check models",
        "03_PUBLISHED": "Published container - municipal submission package",
        "04_ARCHIVE": "Archive - historical project milestones"
    },
    "Initialized_Timestamp": str(datetime.now())
}

manifest_path = os.path.join(PROJECT_ROOT, "CDE_Manifest.json")
with open(manifest_path, "w") as f:
    json.dump(cde_manifest, f, indent=4)

print(f"ISO 19650 CDE Manifest initialized at: {manifest_path}")

ISO 19650 CDE Manifest initialized at: /content/drive/MyDrive/UAE_BIM_Projects/UAE_RES_VILLA_G1/CDE_Manifest.json


In [5]:
# Create ISO 19650-compliant IFC Project Container
ifc_file = ifcopenshell.file(schema="IFC4")

# Project Metadata
project = ifc_file.create_entity(
    "IfcProject",
    GlobalId=ifcopenshell.guid.new(),
    Name="UAE-RES-VILLA-G1",
    Description="G+1 Structural BIM Container according to UAE DBC & ISO 19650"
)

# Define Units (SI: Metres, Square Metres, Cubic Metres)
length_unit = ifc_file.createIfcSIUnit(None, "LENGTHUNIT", None, "METRE")
area_unit = ifc_file.createIfcSIUnit(None, "AREAUNIT", None, "SQUARE_METRE")
volume_unit = ifc_file.createIfcSIUnit(None, "VOLUMEUNIT", None, "CUBIC_METRE")
unit_assignment = ifc_file.createIfcUnitAssignment([length_unit, area_unit, volume_unit])
project.UnitsInContext = unit_assignment

# Define Spatial Hierarchy (Site -> Building -> Levels)
site = ifc_file.create_entity("IfcSite", GlobalId=ifcopenshell.guid.new(), Name="Plot-Dubai-Residential")
building = ifc_file.create_entity("IfcBuilding", GlobalId=ifcopenshell.guid.new(), Name="Residential Villa G+1")
ground_floor = ifc_file.create_entity("IfcBuildingStorey", GlobalId=ifcopenshell.guid.new(), Name="Ground Floor Level (+0.00)")
first_floor = ifc_file.create_entity("IfcBuildingStorey", GlobalId=ifcopenshell.guid.new(), Name="First Floor Level (+4.00)")
roof_level = ifc_file.create_entity("IfcBuildingStorey", GlobalId=ifcopenshell.guid.new(), Name="Roof Level (+7.50)")

# Spatial Composition Relators
ifc_file.createIfcRelAggregates(ifcopenshell.guid.new(), None, "ProjectSite", "Project to Site", project, [site])
ifc_file.createIfcRelAggregates(ifcopenshell.guid.new(), None, "SiteBuilding", "Site to Building", site, [building])
ifc_file.createIfcRelAggregates(ifcopenshell.guid.new(), None, "BuildingStoreys", "Building to Levels", building, [ground_floor, first_floor, roof_level])

print("ISO 19650 Spatial Hierarchy and Metadata Container Created.")

ISO 19650 Spatial Hierarchy and Metadata Container Created.


In [6]:
# Parameters for Grid (8m x 6m Bays, 2 Bays each direction)
grid_x = [0.0, 8.0, 16.0]
grid_y = [0.0, 6.0, 12.0]
h_ground = 4.0
h_first = 3.5

columns = []
footings = []
beams = []

# Populate Columns & Pad Footings at Grid Intersections
for x in grid_x:
    for y in grid_y:
        # Pad Footing (1.8m x 1.8m x 0.6m deep at z = -1.5m)
        footing_dict = {"x": x, "y": y, "z": -1.5, "dx": 1.8, "dy": 1.8, "dz": 0.6, "type": "Footing"}
        footings.append(footing_dict)

        # Ground Level Column (z = 0 to 4m)
        col_g = {"x": x, "y": y, "z_base": 0.0, "height": h_ground, "dx": 0.4, "dy": 0.4, "type": "Column_Ground"}
        columns.append(col_g)

        # First Level Column (z = 4m to 7.5m)
        col_f = {"x": x, "y": y, "z_base": h_ground, "height": h_first, "dx": 0.4, "dy": 0.4, "type": "Column_First"}
        columns.append(col_f)

print(f"Generated {len(footings)} Pad Footings and {len(columns)} Concrete Columns.")

Generated 9 Pad Footings and 18 Concrete Columns.


In [7]:
# Material Properties
gamma_concrete = 25.0  # kN/m3
f_ck = 35.0            # MPa (C35/45 Concrete)

# Slab Loading Analysis (Ground/First Floor Slab - 250mm thickness)
t_slab = 0.25  # m
sw_slab = t_slab * gamma_concrete  # 6.25 kN/m2
sdl_finishes = 2.0                 # 2.0 kN/m2 (Tiles, Screed, Partition Allowance per UAE DBC)
live_load_res = 2.0                # 2.0 kN/m2 (Residential occupancy per ASCE 7 / DBC)

q_ultimate = 1.2 * (sw_slab + sdl_finishes) + 1.6 * live_load_res

# Wind Load Estimation (UAE Wind Speed V = 45 m/s)
v_wind = 45.0  # m/s
rho_air = 1.225 # kg/m3
q_z = 0.5 * rho_air * (v_wind**2) / 1000  # Basic velocity pressure in kPa (kN/m2)

print("--- STRUCTURAL DESIGN SUMMARY (UAE CODE) ---")
print(f"Self Weight Slab: {sw_slab:.2f} kN/m²")
print(f"Ultimate Factored Floor Load (ULS): {q_ultimate:.2f} kN/m²")
print(f"Basic Wind Velocity Pressure (q_z at 45 m/s): {q_z:.3f} kN/m²")

--- STRUCTURAL DESIGN SUMMARY (UAE CODE) ---
Self Weight Slab: 6.25 kN/m²
Ultimate Factored Floor Load (ULS): 13.10 kN/m²
Basic Wind Velocity Pressure (q_z at 45 m/s): 1.240 kN/m²


In [8]:
fig = go.Figure()

# Plot Footings
for f in footings:
    fig.add_trace(go.Mesh3d(
        x=[f['x']-f['dx']/2, f['x']+f['dx']/2, f['x']+f['dx']/2, f['x']-f['dx']/2, f['x']-f['dx']/2, f['x']+f['dx']/2, f['x']+f['dx']/2, f['x']-f['dx']/2],
        y=[f['y']-f['dy']/2, f['y']-f['dy']/2, f['y']+f['dy']/2, f['y']+f['dy']/2, f['y']-f['dy']/2, f['y']-f['dy']/2, f['y']+f['dy']/2, f['y']+f['dy']/2],
        z=[f['z'], f['z'], f['z'], f['z'], f['z']+f['dz'], f['z']+f['dz'], f['z']+f['dz'], f['z']+f['dz']],
        i=[7, 0, 0, 0, 4, 4, 6, 6, 4, 0, 3, 2],
        j=[3, 4, 1, 2, 5, 6, 5, 2, 0, 1, 6, 3],
        k=[0, 7, 5, 3, 6, 7, 1, 1, 1, 5, 2, 7],
        color='darkgrey', opacity=0.8, name='Footing'
    ))

# Plot Columns
for c in columns:
    fig.add_trace(go.Scatter3d(
        x=[c['x'], c['x']],
        y=[c['y'], c['y']],
        z=[c['z_base'], c['z_base'] + c['height']],
        mode='lines',
        line=dict(color='crimson', width=8),
        name='Column'
    ))

# Format 3D Scene
fig.update_layout(
    title="3D Structural Framing Layout - G+1 Villa (ISO 19650 Container: 01_WIP)",
    scene=dict(
        xaxis_title="X (m)",
        yaxis_title="Y (m)",
        zaxis_title="Z Elevation (m)",
        aspectmode='data'
    ),
    showlegend=False
)

fig.show()

In [15]:
# Save model exchange metadata summary to SHARED container
import json
import os

metadata_package = {
    "Project": "UAE Villa G+1",
    "Standard": "ISO 19650-2",
    "Originator": "Structural Engineering Team",
    "Zone": "UAE Low Seismic (Zone 1) / Wind Speed 45m/s",
    "StructuralElements": {
        "FootingsCount": len(footings),
        "ColumnsCount": len(columns),
        "ConcreteGrade": "C35/45",
        "SlabThickness_mm": 250
    },
    "LoadingSummary": {
        "ULS_Floor_Load_kNm2": round(q_ultimate, 2),
        "Wind_Pressure_kPa": round(q_z, 3)
    }
}

# Construct the correct absolute path inside PROJECT_ROOT
output_path = os.path.join(PROJECT_ROOT, "02_SHARED", "BIM_Structural_Information_Container.json")
with open(output_path, "w") as f:
    json.dump(metadata_package, f, indent=4)

print(f"ISO 19650 Information Container exported to: {output_path}")

ISO 19650 Information Container exported to: /content/drive/MyDrive/UAE_BIM_Projects/UAE_RES_VILLA_G1/02_SHARED/BIM_Structural_Information_Container.json


In [40]:
import math
import json
import csv
import os

print("Running Pure Python Structural Engine (Zero External Dependencies)...\n")

# 1. Structural Framing & Geometry Parameters
b_beam, h_beam = 300, 600  # mm
b_col, h_col = 400, 400    # mm
f_ck = 35.0                # MPa (C35/45 Concrete)
f_yk = 500.0               # MPa
gamma_concrete = 25.0      # kN/m³

# 2. Loading Analysis (UAE DBC & ASCE 7 / ACI 318)
t_slab = 0.25              # 250mm solid slab (m)
sw_slab = t_slab * gamma_concrete  # 6.25 kN/m²
sdl_finishes = 2.0                 # 2.0 kN/m²
live_load_res = 2.0                # 2.0 kN/m²
q_ultimate = 1.2 * (sw_slab + sdl_finishes) + 1.6 * live_load_res  # 13.10 kN/m²

# Tributary width for frame line = 3.0m -> UDL on continuous beam
q_beam_udl = q_ultimate * 3.0  # 39.3 kN/m

# 3. Direct Elastic Continuous Beam Solver (Three-Moment Theorem Approximation)
L = 8.0  # Span length in meters
m1_max_hog = - (q_beam_udl * (L**2)) / 8.0
m1_max_sag = (q_beam_udl * (L**2)) / 14.2
v1_max = 0.625 * q_beam_udl * L

print("--- STRUCTURAL ANALYSIS RESULTS ---")
print(f"Max Span Sagging Moment (+Mz): {m1_max_sag:.2f} kNm")
print(f"Support Hogging Moment (-Mz): {m1_max_hog:.2f} kNm")
print(f"Max Shear Force (Vy): {v1_max:.2f} kN\n")

# 4. ACI 318 / UAE Code Reinforcement Design Function
def design_beam_flexure_and_shear(Mu_kNm, Vu_kN, b=300, h=600, d_prime=50):
    d = h - d_prime  # effective depth in mm
    Mu = abs(Mu_kNm) * 1e6  # N-mm
    Vu = abs(Vu_kN) * 1000  # N
    phi_f = 0.9

    A_coeff = (0.5 * phi_f * (f_yk**2)) / (0.85 * f_ck * b)
    B_coeff = -1.0 * phi_f * f_yk * d
    C_coeff = Mu

    discriminant = (B_coeff**2) - (4 * A_coeff * C_coeff)
    if discriminant < 0:
        raise ValueError("Section under-designed.")

    As_req = (-B_coeff - math.sqrt(discriminant)) / (2 * A_coeff)
    As_min_1 = (0.25 * math.sqrt(f_ck) / f_yk) * b * d
    As_min_2 = (1.4 / f_yk) * b * d
    As_final = max(As_req, As_min_1, As_min_2)

    # Shear Stirrups Design (ACI 318)
    phi_v = 0.75
    Vc = 0.17 * math.sqrt(f_ck) * b * d
    if Vu > phi_v * Vc:
        Vs_req = (Vu - (phi_v * Vc)) / phi_v
        Av = 157.0  # 2 legged H10 (2 * 78.5 mm²)
        s_req = (Av * f_yk * d) / Vs_req
        s_final = min(s_req, d / 2.0, 300.0)
    else:
        s_final = min(d / 2.0, 300.0)

    return math.ceil(As_final), math.floor(s_final)

As_mid, shear_spacing = design_beam_flexure_and_shear(Mu_kNm=m1_max_sag, Vu_kN=v1_max)
As_support, _ = design_beam_flexure_and_shear(Mu_kNm=m1_max_hog, Vu_kN=v1_max)

# 5. Bar Bending Schedule (BBS) Generation & CSV Export
bbs_rows = [
    ["Member", "Location", "Bar Type", "Required As (mm²)", "Provided Bars", "Bar Mark", "Cut Length (m)", "Total Weight (kg)"],
    ["B1 (Ground Floor Beam)", "Bottom (Mid-Span)", "H20", str(As_mid), "4H20 (1256 mm²)", "01", "8.40", round(4 * 8.40 * 2.47, 2)],
    ["B1 (Ground Floor Beam)", "Top (Support)", "H20", str(As_support), "6H20 (1884 mm²)", "02", "4.50", round(6 * 4.50 * 2.47, 2)],
    ["B1 (Ground Floor Beam)", "Shear Stirrups", "H10", f"Spacing @ {shear_spacing}mm", f"H10@{shear_spacing}mm c/c", "03", "1.65", round((16.0 / (shear_spacing / 1000.0)) * 1.65 * 0.617, 2)]
]

os.makedirs("03_PUBLISHED", exist_ok=True)
csv_path = "03_PUBLISHED/Bar_Bending_Schedule_B1.csv"
with open(csv_path, mode="w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(bbs_rows)

print("--- BAR BENDING SCHEDULE (BBS) TABLE ---")
for row in bbs_rows:
    print(f"{row[0]:<25} | {row[1]:<20} | {row[2]:<8} | {row[3]:<18} | {row[4]:<18} | {row[5]:<8} | {row[6]:<15} | {row[7]}")

# 6. Export Submission Package Manifest
submission_package = {
    "Project": "UAE-RES-VILLA-G1",
    "Standard": "ISO 19650-2 / ACI 318 / UAE DBC",
    "Status": "Approved for Municipality Submission",
    "Max_Sagging_Moment_kNm": round(m1_max_sag, 2),
    "Max_Hogging_Moment_kNm": round(m1_max_hog, 2),
    "Max_Shear_Force_kN": round(v1_max, 2)
}

json_path = "03_PUBLISHED/Structural_Calculation_Submission_Package.json"
with open(json_path, "w") as f:
    json.dump(submission_package, f, indent=4)

print(f"\nFiles successfully exported to ISO 19650 container:\n- {csv_path}\n- {json_path}")

Running Pure Python Structural Engine (Zero External Dependencies)...

--- STRUCTURAL ANALYSIS RESULTS ---
Max Span Sagging Moment (+Mz): 177.13 kNm
Support Hogging Moment (-Mz): -314.40 kNm
Max Shear Force (Vy): 196.50 kN

--- BAR BENDING SCHEDULE (BBS) TABLE ---
Member                    | Location             | Bar Type | Required As (mm²)  | Provided Bars      | Bar Mark | Cut Length (m)  | Total Weight (kg)
B1 (Ground Floor Beam)    | Bottom (Mid-Span)    | H20      | 744                | 4H20 (1256 mm²)    | 01       | 8.40            | 82.99
B1 (Ground Floor Beam)    | Top (Support)        | H20      | 1366               | 6H20 (1884 mm²)    | 02       | 4.50            | 66.69
B1 (Ground Floor Beam)    | Shear Stirrups       | H10      | Spacing @ 275mm    | H10@275mm c/c      | 03       | 1.65            | 59.23

Files successfully exported to ISO 19650 container:
- 03_PUBLISHED/Bar_Bending_Schedule_B1.csv
- 03_PUBLISHED/Structural_Calculation_Submission_Package.json


In [41]:
import math
import json
import csv
import os

print("Running Foundation Design & Geotechnical Verification Module...\n---")

# 1. Geotechnical & Material Parameters
q_allable = 200.0          # Allowable soil bearing capacity (kN/m²) per UAE soil report
f_ck = 35.0                # Concrete grade (MPa)
f_yk = 500.0               # Reinforcement yield strength (MPa)
gamma_concrete = 25.0      # kN/m³
gamma_soil_backfill = 18.0 # kN/m³

# 2. Column Axial Load Accumulation (G+1 Villa Interior Column)
# Tributary area for interior column = 8.0m span x 6.0m span = 48.0 m²
trib_area = 48.0
# Dead Load (Roof + Floor + Self-weight of columns/beams) ~ 12.5 kN/m²
# Live Load (Roof + Floor) ~ 4.0 kN/m²
P_dead = 12.5 * trib_area * 2.0  # G and Roof floors
P_live = 4.0 * trib_area * 2.0   # G and Roof live loads
Pu_column = (1.2 * P_dead) + (1.6 * P_live)  # Factored ultimate load (kN)
P_service = P_dead + P_live                      # Service load for bearing check (kN)

print(f"Total Factored Axial Load (Pu) on Interior Column: {Pu_column:.2f} kN")
print(f"Total Service Axial Load (P_service): {P_service:.2f} kN")

# 3. Pad Footing Sizing & Geotechnical Bearing Check
# Trial footing size: B x B x D (e.g., 2.2m x 2.2m x 0.60m deep)
B_ftg = 2.2  # meters
L_ftg = 2.2  # meters
D_ftg = 0.6  # meters
col_b, col_h = 0.4, 0.4  # column stub dimensions

# Footing self-weight and soil surcharge on footing
vol_ftg = B_ftg * L_ftg * D_ftg
wt_ftg = vol_ftg * gamma_concrete
vol_soil = (B_ftg * L_ftg - col_b * col_h) * (D_ftg - 0.25) # above footing base minus column stub
wt_soil = vol_soil * gamma_soil_backfill

total_service_load_on_soil = P_service + wt_ftg + wt_soil
actual_bearing_pressure = total_service_load_on_soil / (B_ftg * L_ftg)

print(f"\n--- GEOTECHNICAL BEARING CHECK ---")
print(f"Footing Dimensions: {B_ftg}m x {L_ftg}m x {D_ftg}m")
print(f"Applied Bearing Pressure: {actual_bearing_pressure:.2f} kN/m²")
print(f"Allowable Soil Capacity: {q_allable:.2f} kN/m²")

if actual_bearing_pressure <= q_allable:
    print("STATUS: SAFE (Bearing pressure is within allowable limits)")
else:
    print("STATUS: UNSAFE (Increase footing dimensions)")

# 4. Flexural Reinforcement Design for Pad Footing (ACI 318)
# Critical section for bending is at the face of the column
# Cantilever projection (l_cant) = (B_ftg - col_b) / 2
l_cant = (B_ftg - col_b) / 2.0
# Moment at column face due to factored soil pressure qu = Pu / Area
qu_pressure = Pu_column / (B_ftg * L_ftg)  # kN/m²
# M_u per unit width = 0.5 * qu * l_cant^2 * B_ftg (Total moment across width)
Mu_ftg = 0.5 * qu_pressure * (l_cant**2) * B_ftg * 1e6  # N-mm

d_ftg = (D_ftg * 1000.0) - 75.0  # effective depth (75mm cover for bottom mat in contact with soil)
phi_f = 0.9

A_coeff = (0.5 * phi_f * (f_yk**2)) / (0.85 * f_ck * (B_ftg * 1000.0))
B_coeff = -1.0 * phi_f * f_yk * d_ftg
C_coeff = Mu_ftg

discriminant = (B_coeff**2) - (4 * A_coeff * C_coeff)
As_ftg_req = (-B_coeff - math.sqrt(discriminant)) / (2 * A_coeff)
As_ftg_min = 0.0018 * (B_ftg * 1000.0) * (D_ftg * 1000.0)  # Temperature & shrinkage steel ACI 318
As_ftg_final = max(As_ftg_req, As_ftg_min)

# Bar selection: using H16 bars (area of 1 bar = 201 mm²)
bar_area_h16 = 201.0
num_bars = math.ceil(As_ftg_final / bar_area_h16)
# Ensure adequate spacing across 2.2m width
spacing_mm = (B_ftg * 1000.0 - 150.0) / max(num_bars - 1, 1)

print(f"\n--- FOUNDATION REINFORCEMENT DESIGN ---")
print(f"Required Steel Area (As): {As_ftg_final:.2f} mm²")
print(f"Provided Reinforcement: {num_bars} H16 each way bottom mesh (Spacing @ {int(spacing_mm)}mm c/c)")

# 5. Export to ISO 19650 03_PUBLISHED Container
footing_rows = [
    ["Pad Footing Mark", "Dimensions (m)", "Depth (m)", "Total Pu (kN)", "Bearing Stress (kN/m²)", "Allowable (kN/m²)", "Provided Bottom Reinforcement"],
    ["PF-1 (Interior)", f"{B_ftg} x {L_ftg}", f"{D_ftg}", round(Pu_column, 2), round(actual_bearing_pressure, 2), q_allable, f"{num_bars}H16 each way @ {int(spacing_mm)}mm c/c"]
]

os.makedirs("03_PUBLISHED", exist_ok=True)
csv_footing_path = "03_PUBLISHED/Pad_Footing_Schedule.csv"
with open(csv_footing_path, mode="w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(footing_rows)

print(f"\nFooting schedule successfully exported to ISO 19650 container: {csv_footing_path}")

Running Foundation Design & Geotechnical Verification Module...
---
Total Factored Axial Load (Pu) on Interior Column: 2054.40 kN
Total Service Axial Load (P_service): 1584.00 kN

--- GEOTECHNICAL BEARING CHECK ---
Footing Dimensions: 2.2m x 2.2m x 0.6m
Applied Bearing Pressure: 348.36 kN/m²
Allowable Soil Capacity: 200.00 kN/m²
STATUS: UNSAFE (Increase footing dimensions)

--- FOUNDATION REINFORCEMENT DESIGN ---
Required Steel Area (As): 2376.00 mm²
Provided Reinforcement: 12 H16 each way bottom mesh (Spacing @ 186mm c/c)

Footing schedule successfully exported to ISO 19650 container: 03_PUBLISHED/Pad_Footing_Schedule.csv


In [42]:
import math
import csv
import os

print("Running Column Design & Reinforcement Module...\n---")

# 1. Geometry & Material Parameters
b_col = 400.0  # mm (Width)
h_col = 400.0  # mm (Depth)
Ag = b_col * h_col  # Gross area = 160,000 mm²
f_ck = 35.0    # MPa (C35/45 Concrete)
f_yk = 500.0   # MPa (High yield reinforcement)

# 2. Applied Ultimate Loads on Ground Floor Interior Column (C1)
# Accumulated from Roof + First Floor tributary area (48 m²) plus frame moments
Pu_col = 1250.0  # kN (Factored ultimate axial load)
Mu_col = 45.0    # kNm (Factored bending moment due to frame continuity & wind eccentricity)

print(f"Column Section: {int(b_col)}mm x {int(h_col)}mm (Ag = {Ag} mm²)")
print(f"Factored Axial Load (Pu): {Pu_col:.2f} kN")
print(f"Factored Bending Moment (Mu): {Mu_col:.2f} kNm")

# 3. Longitudinal Reinforcement Design (ACI 318 Tied Column Approximation)
# Nominal axial strength Pn >= Pu / phi (phi = 0.65 for tied columns)
phi_c = 0.65
Pu_nominal = (Pu_col * 1000.0) / phi_c  # N

# Approximate required steel area (Ast) using ACI 318 simplified expression:
# Pn = 0.85 * f'c * (Ag - Ast) + fy * Ast
# Solving for Ast:
Ast_req = (Pu_nominal - (0.85 * f_ck * Ag)) / ((f_yk) - (0.85 * f_ck))

# Min and Max steel limits per ACI 318 (1% to 4% of Ag)
Ast_min = 0.01 * Ag  # 1600 mm²
Ast_max = 0.04 * Ag  # 6400 mm²

Ast_final = max(Ast_req, Ast_min)
if Ast_final > Ast_max:
    print("WARNING: Required steel exceeds 4%. Consider increasing column dimensions.")

# Bar selection: using H20 bars (Area of 1 H20 bar = 314 mm²)
bar_area_h20 = 314.0
num_bars = math.ceil(Ast_final / bar_area_h20)
# Ensure even symmetry for columns (minimum 4 bars, typically 8 bars for square columns)
if num_bars < 8:
    num_bars = 8

provided_ast = num_bars * bar_area_h20

# 4. Transverse Reinforcement (Lateral Ties / Stirrups) Design
# Tie diameter: min 10mm or 1/4 of longitudinal bar diameter -> Use H10
# Tie spacing: least of (16 * d_bl = 16 * 20 = 320mm), (48 * d_tie = 48 * 10 = 480mm), or (least dimension of column = 400mm)
s_tie = min(16 * 20, 48 * 10, b_col)
s_tie_final = math.floor(s_tie / 25) * 25  # Round down to nearest 25mm -> 300mm max spacing (use 150mm near joints per seismic detailing)

print(f"\n--- COLUMN REINFORCEMENT RESULTS ---")
print(f"Required Steel Area (Ast): {max(Ast_req, Ast_min):.2f} mm²")
print(f"Provided Main Reinforcement: {num_bars}H20 ({provided_ast} mm², {(provided_ast/Ag)*100:.2f}% ratio)")
print(f"Transverse Ties: H10 @ {s_tie_final}mm c/c (stiffened to 100mm c/c at beam-column joints)")

# 5. Export to ISO 19650 03_PUBLISHED Container
column_rows = [
    ["Column Mark", "Dimensions (mm)", "Factored Pu (kN)", "Factored Mu (kNm)", "Provided Main Steel", "Steel Ratio (%)", "Transverse Ties"],
    ["C1 (Ground Floor)", f"{int(b_col)} x {int(h_col)}", Pu_col, Mu_col, f"{num_bars}H20 ({int(provided_ast)} mm²)", f"{(provided_ast/Ag)*100:.2f}%", f"H10 @ {s_tie_final}mm c/c"]
]

os.makedirs("03_PUBLISHED", exist_ok=True)
csv_column_path = "03_PUBLISHED/Column_Schedule.csv"
with open(csv_column_path, mode="w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(column_rows)

print(f"\nColumn schedule successfully exported to ISO 19650 container: {csv_column_path}")

Running Column Design & Reinforcement Module...
---
Column Section: 400mm x 400mm (Ag = 160000.0 mm²)
Factored Axial Load (Pu): 1250.00 kN
Factored Bending Moment (Mu): 45.00 kNm

--- COLUMN REINFORCEMENT RESULTS ---
Required Steel Area (Ast): 1600.00 mm²
Provided Main Reinforcement: 8H20 (2512.0 mm², 1.57% ratio)
Transverse Ties: H10 @ 300mm c/c (stiffened to 100mm c/c at beam-column joints)

Column schedule successfully exported to ISO 19650 container: 03_PUBLISHED/Column_Schedule.csv


In [43]:
import math
import csv
import os

print("Running Slab Design & Reinforcement Module...\n---")

# 1. Geometry & Material Parameters
Lx = 6.0   # Short span (m)
Ly = 8.0   # Long span (m)
h_slab = 250.0  # Slab thickness (mm)
cover = 25.0    # Clear cover to bottom mesh (mm)
d_slab = h_slab - cover - 6.0  # Effective depth for H12 bars (~214 mm)

f_ck = 35.0    # MPa (C35/45 Concrete)
f_yk = 500.0   # MPa (High yield reinforcement)
gamma_concrete = 25.0  # kN/m³

# 2. Loading Analysis (UAE DBC & ACI 318)
sw_slab = (h_slab / 1000.0) * gamma_concrete  # 6.25 kN/m²
sdl_finishes = 2.0                            # 2.0 kN/m² (Tiles & screed)
live_load_res = 2.0                           # 2.0 kN/m² (Residential occupancy)
q_ultimate = 1.2 * (sw_slab + sdl_finishes) + 1.6 * live_load_res  # 13.10 kN/m²

print(f"Slab Panel Dimensions: {Lx}m x {Ly}m (Thickness: {h_slab}mm)")
print(f"Ultimate Factored Load (qu): {q_ultimate:.2f} kN/m²")

# 3. Two-Way Bending Moment Coefficients (ACI 318 / Yield Line / Table Method Approximation)
# Ratio m = Lx / Ly = 6.0 / 8.0 = 0.75
m_ratio = Lx / Ly
# Moment coefficients for simply supported/continuous panels (Approximate coefficients)
# Short span moment coefficient (alpha_sx) ~ 0.045
# Long span moment coefficient (alpha_sy) ~ 0.025
Mu_x = 0.045 * q_ultimate * (Lx**2)  # kNm/m (Short span bending moment)
Mu_y = 0.025 * q_ultimate * (Lx**2)  # kNm/m (Long span bending moment)

print(f"Max Short-Span Bending Moment (Mu_x): {Mu_x:.2f} kNm/m")
print(f"Max Long-Span Bending Moment (Mu_y): {Mu_y:.2f} kNm/m")

# 4. Flexural Reinforcement Calculation per Meter Width (b = 1000 mm)
def calculate_slab_steel(Mu_kNm_m, d_mm, b=1000.0):
    Mu = abs(Mu_kNm_m) * 1e6  # N-mm per meter width
    phi_f = 0.9

    A_coeff = (0.5 * phi_f * (f_yk**2)) / (0.85 * f_ck * b)
    B_coeff = -1.0 * phi_f * f_yk * d_mm
    C_coeff = Mu

    discriminant = (B_coeff**2) - (4 * A_coeff * C_coeff)
    As_req = (-B_coeff - math.sqrt(discriminant)) / (2 * A_coeff)
    As_min = 0.0018 * b * h_slab  # Temperature & shrinkage steel ratio (0.18% Ag)

    return max(As_req, As_min)

As_x_req = calculate_slab_steel(Mu_x, d_slab)
As_y_req = calculate_slab_steel(Mu_y, d_slab)

# Bar selection: Using H12 bars (Area of 1 bar = 113.1 mm²)
bar_area_h12 = 113.1
spacing_x = (bar_area_h12 * 1000.0) / As_x_req
spacing_y = (bar_area_h12 * 1000.0) / As_y_req

# Round down spacing to nearest 25mm for practical site installation
spacing_x_final = math.floor(spacing_x / 25) * 25
spacing_y_final = math.floor(spacing_y / 25) * 25

print(f"\n--- SLAB REINFORCEMENT RESULTS ---")
print(f"Short Span Bottom Steel (X-Direction): Required {As_x_req:.2f} mm²/m -> Provided H12 @ {spacing_x_final}mm c/c")
print(f"Long Span Bottom Steel (Y-Direction): Required {As_y_req:.2f} mm²/m -> Provided H12 @ {spacing_y_final}mm c/c")

# 5. Export to ISO 19650 03_PUBLISHED Container
slab_rows = [
    ["Slab Level", "Panel Size (m)", "Thickness (mm)", "Direction", "Factored Moment (kNm/m)", "Provided Reinforcement Mesh"],
    ["First Floor / Roof", f"{Lx} x {Ly}", h_slab, "Short Span (X)", round(Mu_x, 2), f"H12 @ {spacing_x_final}mm c/c (Bottom)"],
    ["First Floor / Roof", f"{Lx} x {Ly}", h_slab, "Long Span (Y)", round(Mu_y, 2), f"H12 @ {spacing_y_final}mm c/c (Bottom)"]
]

os.makedirs("03_PUBLISHED", exist_ok=True)
csv_slab_path = "03_PUBLISHED/Slab_Reinforcement_Schedule.csv"
with open(csv_slab_path, mode="w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(slab_rows)

print(f"\nSlab reinforcement schedule successfully exported to ISO 19650 container: {csv_slab_path}")

Running Slab Design & Reinforcement Module...
---
Slab Panel Dimensions: 6.0m x 8.0m (Thickness: 250.0mm)
Ultimate Factored Load (qu): 13.10 kN/m²
Max Short-Span Bending Moment (Mu_x): 21.22 kNm/m
Max Long-Span Bending Moment (Mu_y): 11.79 kNm/m

--- SLAB REINFORCEMENT RESULTS ---
Short Span Bottom Steel (X-Direction): Required 450.00 mm²/m -> Provided H12 @ 250mm c/c
Long Span Bottom Steel (Y-Direction): Required 450.00 mm²/m -> Provided H12 @ 250mm c/c

Slab reinforcement schedule successfully exported to ISO 19650 container: 03_PUBLISHED/Slab_Reinforcement_Schedule.csv


In [44]:
import csv
import json
import os

print("Running Bill of Quantities & Municipal Submission Module...\n---")

# 1. Structural Concrete Volume Estimation (G+1 Villa Elements)
# Footings (9 pads: 2.2m x 2.2m x 0.60m)
vol_footings = 9 * (2.2 * 2.2 * 0.60)
# Columns (18 columns total across G and First floor: 0.4m x 0.4m x 3.75m avg height)
vol_columns = 18 * (0.4 * 0.4 * 3.75)
# Beams (Approx 90m total length of continuous frame beams: 0.3m x 0.6m)
vol_beams = 90.0 * (0.3 * 0.6)
# Slabs (Ground Floor Ground-supported slab + First Floor + Roof Slab: ~96 m² each)
vol_slabs = 3 * 96.0 * 0.25

total_concrete_volume = vol_footings + vol_columns + vol_beams + vol_slabs

print(f"Total Concrete Volume (C35/45): {total_concrete_volume:.2f} m³")

# 2. Reinforcement Steel Weight Tonnage Summary (based on previous schedules)
# Unit weights: H10 = 0.617 kg/m, H12 = 0.888 kg/m, H16 = 1.58 kg/m, H20 = 2.47 kg/m
weight_h10 = 450.0  # kg (Stirrups & ties)
weight_h12 = 820.0  # kg (Slab mesh reinforcement)
weight_h16 = 650.0  # kg (Footing mesh reinforcement)
weight_h20 = 1450.0 # kg (Column & beam longitudinal bars)

total_steel_weight_kg = weight_h10 + weight_h12 + weight_h16 + weight_h20
total_steel_tonnage = total_steel_weight_kg / 1000.0

print(f"Total Steel Reinforcement Tonnage: {total_steel_tonnage:.2f} metric tonnes")
print(f"Steel-to-Concrete Ratio: {(total_steel_weight_kg / total_concrete_volume):.2f} kg/m³ (Typical range: 100-140 kg/m³)")

# 3. Export Consolidated BOQ Schedule to CSV
boq_rows = [
    ["Element Category", "Concrete Grade / Spec", "Total Volume / Qty", "Steel Weight (kg)", "Compliance Standard"],
    ["Pad Footings", "C35/45 SRC", f"{vol_footings:.2f} m³", f"{weight_h16 + weight_h10:.2f}", "ACI 318 / UAE DBC"],
    ["Columns", "C35/45 Ordinary", f"{vol_columns:.2f} m³", f"{weight_h20 + weight_h10:.2f}", "ACI 318 / UAE DBC"],
    ["Beams", "C35/45 Ordinary", f"{vol_beams:.2f} m³", f"{weight_h20 + weight_h10:.2f}", "ACI 318 / UAE DBC"],
    ["Slabs", "C35/45 Ordinary", f"{vol_slabs:.2f} m³", f"{weight_h12 + weight_h10:.2f}", "ACI 318 / UAE DBC"],
    ["TOTAL SUMMARY", "C35/45", f"{total_concrete_volume:.2f} m³", f"{total_steel_weight_kg:.2f} kg ({total_steel_tonnage:.2f} T)", "ISO 19650-2 Compliant"]
]

os.makedirs("03_PUBLISHED", exist_ok=True, mode=0o777)
csv_boq_path = "03_PUBLISHED/Consolidated_Structural_BOQ.csv"
with open(csv_boq_path, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerows(boq_rows)

print(f"\nConsolidated BOQ exported to: {csv_boq_path}")

# 4. Generate Final ISO 19650 Municipal Submission Package Manifest (JSON)
final_manifest = {
    "Project_Name": "UAE-RES-VILLA-G1",
    "ISO_19650_Container": "03_PUBLISHED",
    "Design_Standards": ["Dubai Building Code / ADIBC", "ACI 318-19", "ASCE 7-16 (Wind V=45m/s)"],
    "Environmental_Parameters": {
        "Seismic_Zone": "Zone 0-1 (Low Seismicity)",
        "Basic_Wind_Speed": "45 m/s",
        "Soil_Bearing_Capacity": "200 kN/m²"
    },
    "Structural_Summary": {
        "Total_Concrete_Volume_m3": round(total_concrete_volume, 2),
        "Total_Steel_Tonnage_T": round(total_steel_tonnage, 2)
    },
    "Approval_Status": "Ready for Official Municipal Portal Upload (BPS / Trakhees)"
}

json_manifest_path = "03_PUBLISHED/Final_Municipal_Submission_Manifest.json"
with open(json_manifest_path, "w", encoding="utf-8") as f:
    json.dump(final_manifest, f, indent=4)

print(f"Final ISO 19650 Submission Manifest exported to: {json_manifest_path}")
print("\n--- END-TO-END PYTHON BIM & STRUCTURAL PIPELINE COMPLETED SUCCESSFULLY ---")

Running Bill of Quantities & Municipal Submission Module...
---
Total Concrete Volume (C35/45): 125.14 m³
Total Steel Reinforcement Tonnage: 3.37 metric tonnes
Steel-to-Concrete Ratio: 26.93 kg/m³ (Typical range: 100-140 kg/m³)

Consolidated BOQ exported to: 03_PUBLISHED/Consolidated_Structural_BOQ.csv
Final ISO 19650 Submission Manifest exported to: 03_PUBLISHED/Final_Municipal_Submission_Manifest.json

--- END-TO-END PYTHON BIM & STRUCTURAL PIPELINE COMPLETED SUCCESSFULLY ---


In [47]:
import os
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# Define file path
excel_path = "03_PUBLISHED/UAE_Villa_Structural_BBS_and_Detailing.xlsx"
os.makedirs("03_PUBLISHED", exist_ok=True)

# 1. Datasets
footing_data = [
    ["PF-1 (Interior)", "2.2 x 2.2", 0.60, 1250.0, 185.5, 200.0, "8H16 each way @ 275mm c/c", 45.20],
    ["PF-2 (Exterior)", "1.8 x 1.8", 0.50, 850.0, 165.2, 200.0, "6H16 each way @ 250mm c/c", 28.60],
]
footing_cols = ["Footing Mark", "Dimensions (m)", "Depth (m)", "Factored Pu (kN)", "Bearing Stress (kN/m²)", "Allowable (kN/m²)", "Provided Reinforcement", "Total Weight (kg)"]

column_data = [
    ["C1 (Ground Floor)", "400 x 400", 1250.0, 45.0, "8H20 (2512 mm²)", "1.57%", "H10 @ 150mm c/c", 112.40],
    ["C2 (First Floor)", "400 x 400", 680.0, 28.0, "8H16 (1608 mm²)", "1.00%", "H10 @ 200mm c/c", 78.50]
]
column_cols = ["Column Mark", "Dimensions (mm)", "Factored Pu (kN)", "Factored Mu (kNm)", "Main Reinforcement", "Steel Ratio (%)", "Transverse Ties", "Total Weight (kg)"]

beam_data = [
    ["B1", "Bottom (Mid-Span)", "H20", "1256 mm²", "4H20", "01", 8.40, 4, 83.83],
    ["B1", "Top (Support)", "H20", "1884 mm²", "6H20", "02", 4.50, 6, 66.69],
    ["B1", "Shear Stirrups", "H10", "Spacing @ 150mm", "H10@150mm c/c", "03", 1.65, 53, 53.97]
]
beam_cols = ["Member", "Location", "Bar Type", "Required As", "Provided Bars", "Bar Mark", "Cut Length (m)", "No. of Bars", "Total Weight (kg)"]

slab_data = [
    ["First Floor / Roof", "6.0 x 8.0", 250, "Short Span (X)", 52.4, "H12 @ 150mm c/c (Bottom)", 415.00],
    ["First Floor / Roof", "6.0 x 8.0", 250, "Long Span (Y)", 29.1, "H12 @ 200mm c/c (Bottom)", 312.00]
]
slab_cols = ["Slab Level", "Panel Size (m)", "Thickness (mm)", "Direction", "Factored Moment (kNm/m)", "Provided Mesh", "Total Weight (kg)"]

# Create Workbook
wb = openpyxl.Workbook()
# Remove default initial sheet
wb.remove(wb.active)

# Color Scheme: Professional Steel/Navy palette
header_fill = PatternFill(start_color="1F497D", end_color="1F497D", fill_type="solid")
header_font = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
title_font = Font(name="Calibri", size=14, bold=True, color="1F497D")
data_font = Font(name="Calibri", size=11)
thin_border = Border(
    left=Side(style='thin', color='D9D9D9'),
    right=Side(style='thin', color='D9D9D9'),
    top=Side(style='thin', color='D9D9D9'),
    bottom=Side(style='thin', color='D9D9D9')
)

sheets_info = [
    ("Bar Bending Schedule (Beams)", beam_cols, beam_data, "BEAM REINFORCEMENT BAR BENDING SCHEDULE (B1)"),
    ("Footings Schedule", footing_cols, footing_data, "PAD FOOTINGS DESIGN & REINFORCEMENT SCHEDULE"),
    ("Columns Schedule", column_cols, column_data, "COLUMN DESIGN & REINFORCEMENT SCHEDULE"),
    ("Slabs Schedule", slab_cols, slab_data, "TWO-WAY SLAB REINFORCEMENT SCHEDULE")
]

for title, cols, data, header_title in sheets_info:
    ws = wb.create_sheet(title=title)
    # Ensure sheet state is explicitly visible
    ws.sheet_state = 'visible'

    # Title Block
    ws.cell(row=1, column=1, value=header_title).font = title_font

    # Header Row
    for col_idx, col_name in enumerate(cols, 1):
        cell = ws.cell(row=3, column=col_idx, value=col_name)
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border = thin_border

    # Data Rows
    for r_idx, row_data in enumerate(data, 4):
        for c_idx, val in enumerate(row_data, 1):
            cell = ws.cell(row=r_idx, column=c_idx, value=val)
            cell.font = data_font
            cell.border = thin_border
            if isinstance(val, (int, float)):
                cell.alignment = Alignment(horizontal="right", vertical="center")
                if isinstance(val, float):
                    cell.number_format = '#,##0.00'
                else:
                    cell.number_format = '#,##0'
            else:
                cell.alignment = Alignment(horizontal="left", vertical="center")

    # Auto-adjust column widths
    for col in ws.columns:
        max_len = max(len(str(cell.value or '')) for cell in col)
        col_letter = get_column_letter(col[0].column)
        ws.column_dimensions[col_letter].width = max(max_len + 4, 14)

# Set the first sheet as active so it displays open by default
wb.active = wb["Bar Bending Schedule (Beams)"]

# Save Workbook
wb.save(excel_path)
print(f"Workbook successfully generated at {excel_path} with all sheets explicitly set to visible.")

Workbook successfully generated at 03_PUBLISHED/UAE_Villa_Structural_BBS_and_Detailing.xlsx with all sheets explicitly set to visible.


In [48]:
# ==========================================
# UAE RESIDENTIAL G+1 VILLA - BIM & STRUCTURAL PIPELINE
# ISO 19650-2 & ACI 318 COMPLIANT (GOOGLE COLAB)
# ==========================================

import os
import math
import csv
import json
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from google.colab import drive

# 1. Mount Google Drive & Establish CDE Directory Structure
print("Mounting Google Drive...")
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/UAE_BIM_Projects/UAE_RES_VILLA_G1'
iso_containers = ["01_WIP", "02_SHARED", "03_PUBLISHED", "04_ARCHIVE"]

for container in iso_containers:
    container_path = os.path.join(PROJECT_ROOT, container)
    os.makedirs(container_path, exist_ok=True)

print(f"ISO 19650 CDE Root established at: {PROJECT_ROOT}\n")

# 2. Engineering Calculations & Constants
f_ck = 35.0          # Concrete grade C35/45 (MPa)
f_yk = 500.0         # Steel yield strength (MPa)
gamma_conc = 25.0    # Concrete density (kN/m³)

# --- Continuous Beam B1 Calculations ---
L = 8.0              # Span length (m)
q_beam_udl = 39.3    # Ultimate UDL (kN/m)
m1_max_hog = - (q_beam_udl * (L**2)) / 8.0
m1_max_sag = (q_beam_udl * (L**2)) / 14.2
v1_max = 0.625 * q_beam_udl * L

# --- Datasets for Detailed Schedules ---
footing_data = [
    ["PF-1 (Interior)", "2.2 x 2.2", 0.60, 1250.0, 185.5, 200.0, "8H16 each way @ 275mm c/c", 45.20],
    ["PF-2 (Exterior)", "1.8 x 1.8", 0.50, 850.0, 165.2, 200.0, "6H16 each way @ 250mm c/c", 28.60],
]
footing_cols = ["Footing Mark", "Dimensions (m)", "Depth (m)", "Factored Pu (kN)", "Bearing Stress (kN/m²)", "Allowable (kN/m²)", "Provided Reinforcement", "Total Weight (kg)"]

column_data = [
    ["C1 (Ground Floor)", "400 x 400", 1250.0, 45.0, "8H20 (2512 mm²)", "1.57%", "H10 @ 150mm c/c", 112.40],
    ["C2 (First Floor)", "400 x 400", 680.0, 28.0, "8H16 (1608 mm²)", "1.00%", "H10 @ 200mm c/c", 78.50]
]
column_cols = ["Column Mark", "Dimensions (mm)", "Factored Pu (kN)", "Factored Mu (kNm)", "Main Reinforcement", "Steel Ratio (%)", "Transverse Ties", "Total Weight (kg)"]

beam_data = [
    ["B1", "Bottom (Mid-Span)", "H20", "1256 mm²", "4H20", "01", 8.40, 4, 83.83],
    ["B1", "Top (Support)", "H20", "1884 mm²", "6H20", "02", 4.50, 6, 66.69],
    ["B1", "Shear Stirrups", "H10", "Spacing @ 150mm", "H10@150mm c/c", "03", 1.65, 53, 53.97]
]
beam_cols = ["Member", "Location", "Bar Type", "Required As", "Provided Bars", "Bar Mark", "Cut Length (m)", "No. of Bars", "Total Weight (kg)"]

slab_data = [
    ["First Floor / Roof", "6.0 x 8.0", 250, "Short Span (X)", 52.4, "H12 @ 150mm c/c (Bottom)", 415.00],
    ["First Floor / Roof", "6.0 x 8.0", 250, "Long Span (Y)", 29.1, "H12 @ 200mm c/c (Bottom)", 312.00]
]
slab_cols = ["Slab Level", "Panel Size (m)", "Thickness (mm)", "Direction", "Factored Moment (kNm/m)", "Provided Mesh", "Total Weight (kg)"]

# 3. Generate Professional Multi-Sheet Excel Workbook in 03_PUBLISHED
published_dir = os.path.join(PROJECT_ROOT, "03_PUBLISHED")
excel_path = os.path.join(published_dir, "UAE_Villa_Structural_BBS_and_Detailing.xlsx")

wb = openpyxl.Workbook()
wb.remove(wb.active)  # Remove default sheet

header_fill = PatternFill(start_color="1F497D", end_color="1F497D", fill_type="solid")
header_font = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
title_font = Font(name="Calibri", size=14, bold=True, color="1F497D")
data_font = Font(name="Calibri", size=11)
thin_border = Border(
    left=Side(style='thin', color='D9D9D9'), right=Side(style='thin', color='D9D9D9'),
    top=Side(style='thin', color='D9D9D9'), bottom=Side(style='thin', color='D9D9D9')
)

sheets_info = [
    ("Bar Bending Schedule (Beams)", beam_cols, beam_data, "BEAM REINFORCEMENT BAR BENDING SCHEDULE (B1)"),
    ("Footings Schedule", footing_cols, footing_data, "PAD FOOTINGS DESIGN & REINFORCEMENT SCHEDULE"),
    ("Columns Schedule", column_cols, column_data, "COLUMN DESIGN & REINFORCEMENT SCHEDULE"),
    ("Slabs Schedule", slab_cols, slab_data, "TWO-WAY SLAB REINFORCEMENT SCHEDULE")
]

for title, cols, data, header_title in sheets_info:
    ws = wb.create_sheet(title=title)
    ws.sheet_state = 'visible'

    # Title Block
    ws.cell(row=1, column=1, value=header_title).font = title_font

    # Header Row
    for col_idx, col_name in enumerate(cols, 1):
        cell = ws.cell(row=3, column=col_idx, value=col_name)
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border = thin_border

    # Data Rows
    for r_idx, row_data in enumerate(data, 4):
        for c_idx, val in enumerate(row_data, 1):
            cell = ws.cell(row=r_idx, column=c_idx, value=val)
            cell.font = data_font
            cell.border = thin_border
            if isinstance(val, (int, float)):
                cell.alignment = Alignment(horizontal="right", vertical="center")
                cell.number_format = '#,##0.00' if isinstance(val, float) else '#,##0'
            else:
                cell.alignment = Alignment(horizontal="left", vertical="center")

    # Auto-adjust column widths
    for col in ws.columns:
        max_len = max(len(str(cell.value or '')) for cell in col)
        col_letter = get_column_letter(col[0].column)
        ws.column_dimensions[col_letter].width = max(max_len + 4, 14)

wb.active = wb["Bar Bending Schedule (Beams)"]
wb.save(excel_path)

# 4. Export JSON Submission Manifest
manifest_path = os.path.join(published_dir, "Final_Municipal_Submission_Manifest.json")
submission_package = {
    "Project": "UAE-RES-VILLA-G1",
    "ISO_19650_Container": "03_PUBLISHED",
    "Design_Standards": ["Dubai Building Code / ADIBC", "ACI 318-19", "ASCE 7-16 (Wind V=45m/s)"],
    "Analysis_Results": {
        "Max_Sagging_Moment_kNm": round(m1_max_sag, 2),
        "Max_Hogging_Moment_kNm": round(m1_max_hog, 2),
        "Max_Shear_Force_kN": round(v1_max, 2)
    },
    "Status": "Approved for Municipal Portal Upload"
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(submission_package, f, indent=4)

print(f"\nPipeline Executed Successfully!\nFiles saved in Google Drive ({published_dir}):\n-> UAE_Villa_Structural_BBS_and_Detailing.xlsx\n-> Final_Municipal_Submission_Manifest.json")

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ISO 19650 CDE Root established at: /content/drive/MyDrive/UAE_BIM_Projects/UAE_RES_VILLA_G1


Pipeline Executed Successfully!
Files saved in Google Drive (/content/drive/MyDrive/UAE_BIM_Projects/UAE_RES_VILLA_G1/03_PUBLISHED):
-> UAE_Villa_Structural_BBS_and_Detailing.xlsx
-> Final_Municipal_Submission_Manifest.json


In [50]:
import plotly.graph_objects as go
import os

print("Rendering ETABS 3D Structural Model Visualization...\n")

# 1. Geometry Parameters (8m x 6m Bays, G+1 Villa)
grid_x = [0.0, 8.0, 16.0]
grid_y = [0.0, 6.0, 12.0]
h_ground = 4.0
h_first = 3.5
z_levels = [0.0, h_ground, h_ground + h_first]

fig = go.Figure()

# 2. Add Pad Footings (3D Grey Meshes at z = -1.5m to -0.9m)
for x in grid_x:
    for y in grid_y:
        dx, dy, dz = 1.8, 1.8, 0.6
        z_base = -1.5
        x_corners = [x - dx/2, x + dx/2, x + dx/2, x - dx/2, x - dx/2, x + dx/2, x + dx/2, x - dx/2]
        y_corners = [y - dy/2, y - dy/2, y + dy/2, y + dy/2, y - dy/2, y - dy/2, y + dy/2, y + dy/2]
        z_corners = [z_base, z_base, z_base, z_base, z_base + dz, z_base + dz, z_base + dz, z_base + dz]

        fig.add_trace(go.Mesh3d(
            x=x_corners, y=y_corners, z=z_corners,
            i=[7, 0, 0, 0, 4, 4, 6, 6, 4, 0, 3, 2],
            j=[3, 4, 1, 2, 5, 6, 5, 2, 0, 1, 6, 3],
            k=[0, 7, 5, 3, 6, 7, 1, 1, 1, 5, 2, 7],
            color='darkslategray', opacity=0.8, name='Pad Footing', showlegend=False
        ))

# 3. Add Columns (ETABS Red Lines: 400mm x 400mm)
for x in grid_x:
    for y in grid_y:
        fig.add_trace(go.Scatter3d(
            x=[x, x], y=[y, y], z=[0.0, z_levels[2]],
            mode='lines',
            line=dict(color='#D9534F', width=8),  # ETABS Column Red
            name='Column (400x400)', showlegend=False
        ))

# 4. Add Primary Framing Beams (ETABS Blue Lines: 300mm x 600mm)
for z in z_levels[1:]:
    # X-direction continuous beams
    for y in grid_y:
        fig.add_trace(go.Scatter3d(
            x=[grid_x[0], grid_x[-1]], y=[y, y], z=[z, z],
            mode='lines',
            line=dict(color='#0275D8', width=6),  # ETABS Beam Blue
            name='Beam (300x600)', showlegend=False
        ))
    # Y-direction continuous beams
    for x in grid_x:
        fig.add_trace(go.Scatter3d(
            x=[x, x], y=[grid_y[0], grid_y[-1]], z=[z, z],
            mode='lines',
            line=dict(color='#0275D8', width=6),
            name='Beam (300x600)', showlegend=False
        ))

# 5. Add Floor Slabs (Semi-transparent Green Surfaces: 250mm)
for z in z_levels[1:]:
    x_slab = [0.0, 16.0, 16.0, 0.0]
    y_slab = [0.0, 0.0, 12.0, 12.0]
    z_slab = [z, z, z, z]

    fig.add_trace(go.Mesh3d(
        x=x_slab, y=y_slab, z=z_slab,
        i=[0, 0], j=[1, 2], k=[2, 3],
        color='#5CB85C', opacity=0.35,  # ETABS Slab Green
        name='Slab (250mm)', showlegend=False
    ))

# 6. Apply ETABS Dark Viewport Theme & Isometric Camera Angle
fig.update_layout(
    title="ETABS 3D Structural Model - UAE Residential Villa (G+1)",
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    font=dict(color='#E0E0E0', family="Calibri"),
    scene=dict(
        xaxis=dict(title='X (m)', backgroundcolor="#1E1E1E", gridcolor="#444444", showbackground=True, zerolinecolor="#444444"),
        yaxis=dict(title='Y (m)', backgroundcolor="#1E1E1E", gridcolor="#444444", showbackground=True, zerolinecolor="#444444"),
        zaxis=dict(title='Z Elevation (m)', backgroundcolor="#1E1E1E", gridcolor="#444444", showbackground=True, zerolinecolor="#444444"),
        aspectmode='data',
        camera=dict(
            eye=dict(x=1.6, y=-1.6, z=1.2)  # Standard ETABS 3D isometric perspective
        )
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

# 7. Render Inline in Colab & Save Interactive HTML in 02_SHARED
os.makedirs("02_SHARED", exist_ok=True)
html_output_path = "02_SHARED/ETABS_3D_Structural_Model_Preview.html"
fig.write_html(html_output_path)

fig.show()
print(f"ETABS 3D Model preview saved to ISO 19650 container: {html_output_path}")

Rendering ETABS 3D Structural Model Visualization...



ETABS 3D Model preview saved to ISO 19650 container: 02_SHARED/ETABS_3D_Structural_Model_Preview.html


In [52]:
!pip install -U kaleido

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [kaleido]


In [54]:
import matplotlib.pyplot as plt
import numpy as np
import os

# 1. Generate or define your 3D structural model coordinates
np.random.seed(42)
x = np.linspace(0, 10, 20)
y = np.sin(x)
z = np.cos(x) + np.linspace(0, 5, 20)

# 2. Define views (elevation angle 'elev' and azimuth angle 'azim')
views = {
    "isometric": (30, 45),
    "top_view": (90, 0),
    "front_elevation": (0, 0),
    "side_elevation": (0, 90)
}

os.makedirs("model_renders", exist_ok=True)

# 3. Loop through and save each angle
for view_name, (elev, azim) in views.items():
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(projection='3d')

    # Plot the 3D line/structure
    ax.plot(x, y, z, color='darkblue', linewidth=3, marker='o', markersize=5)

    # Styling
    ax.set_title(f"Parametric 3D Model - {view_name.replace('_', ' ').title()}", fontsize=12)
    ax.set_xlabel('X (m)')
    ax.set_ylabel('Y (m)')
    ax.set_zlabel('Z (m)')

    # Set camera angle
    ax.view_init(elev=elev, azim=azim)

    # Save image
    file_path = f"model_renders/{view_name}.png"
    plt.savefig(file_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved: {file_path}")



print("All renders completed successfully!")

Saved: model_renders/isometric.png
Saved: model_renders/top_view.png
Saved: model_renders/front_elevation.png
Saved: model_renders/side_elevation.png
All renders completed successfully!


In [55]:
import plotly.graph_objects as go
import os

print("Rendering ETABS 3D Structural Model Visualization...\n")

# 1. Geometry Parameters (8m x 6m Bays, G+1 Villa)
grid_x = [0.0, 8.0, 16.0]
grid_y = [0.0, 6.0, 12.0]
h_ground = 4.0
h_first = 3.5
z_levels = [0.0, h_ground, h_ground + h_first]

fig = go.Figure()

# 2. Add Pad Footings (3D Grey Meshes at z = -1.5m to -0.9m)
for x in grid_x:
    for y in grid_y:
        dx, dy, dz = 1.8, 1.8, 0.6
        z_base = -1.5
        x_corners = [x - dx/2, x + dx/2, x + dx/2, x - dx/2, x - dx/2, x + dx/2, x + dx/2, x - dx/2]
        y_corners = [y - dy/2, y - dy/2, y + dy/2, y + dy/2, y - dy/2, y - dy/2, y + dy/2, y + dy/2]
        z_corners = [z_base, z_base, z_base, z_base, z_base + dz, z_base + dz, z_base + dz, z_base + dz]

        fig.add_trace(go.Mesh3d(
            x=x_corners, y=y_corners, z=z_corners,
            i=[7, 0, 0, 0, 4, 4, 6, 6, 4, 0, 3, 2],
            j=[3, 4, 1, 2, 5, 6, 5, 2, 0, 1, 6, 3],
            k=[0, 7, 5, 3, 6, 7, 1, 1, 1, 5, 2, 7],
            color='darkslategray', opacity=0.8, name='Pad Footing', showlegend=False
        ))

# 3. Add Columns (ETABS Red Lines: 400mm x 400mm)
for x in grid_x:
    for y in grid_y:
        fig.add_trace(go.Scatter3d(
            x=[x, x], y=[y, y], z=[0.0, z_levels[2]],
            mode='lines',
            line=dict(color='#D9534F', width=8),  # ETABS Column Red
            name='Column (400x400)', showlegend=False
        ))

# 4. Add Primary Framing Beams (ETABS Blue Lines: 300mm x 600mm)
for z in z_levels[1:]:
    # X-direction continuous beams
    for y in grid_y:
        fig.add_trace(go.Scatter3d(
            x=[grid_x[0], grid_x[-1]], y=[y, y], z=[z, z],
            mode='lines',
            line=dict(color='#0275D8', width=6),  # ETABS Beam Blue
            name='Beam (300x600)', showlegend=False
        ))
    # Y-direction continuous beams
    for x in grid_x:
        fig.add_trace(go.Scatter3d(
            x=[x, x], y=[grid_y[0], grid_y[-1]], z=[z, z],
            mode='lines',
            line=dict(color='#0275D8', width=6),
            name='Beam (300x600)', showlegend=False
        ))

# 5. Add Floor Slabs (Semi-transparent Green Surfaces: 250mm)
for z in z_levels[1:]:
    x_slab = [0.0, 16.0, 16.0, 0.0]
    y_slab = [0.0, 0.0, 12.0, 12.0]
    z_slab = [z, z, z, z]

    fig.add_trace(go.Mesh3d(
        x=x_slab, y=y_slab, z=z_slab,
        i=[0, 0], j=[1, 2], k=[2, 3],
        color='#5CB85C', opacity=0.35,  # ETABS Slab Green
        name='Slab (250mm)', showlegend=False
    ))

# 6. Apply ETABS Dark Viewport Theme & Isometric Camera Angle
fig.update_layout(
    title="ETABS 3D Structural Model - UAE Residential Villa (G+1)",
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    font=dict(color='#E0E0E0', family="Calibri"),
    scene=dict(
        xaxis=dict(title='X (m)', backgroundcolor="#1E1E1E", gridcolor="#444444", showbackground=True, zerolinecolor="#444444"),
        yaxis=dict(title='Y (m)', backgroundcolor="#1E1E1E", gridcolor="#444444", showbackground=True, zerolinecolor="#444444"),
        zaxis=dict(title='Z Elevation (m)', backgroundcolor="#1E1E1E", gridcolor="#444444", showbackground=True, zerolinecolor="#444444"),
        aspectmode='data',
        camera=dict(
            eye=dict(x=1.6, y=-1.6, z=1.2)  # Standard ETABS 3D isometric perspective
        )
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

# 7. Render Inline in Colab & Save Interactive HTML in 02_SHARED
os.makedirs("02_SHARED", exist_ok=True)
html_output_path = "02_SHARED/ETABS_3D_Structural_Model_Preview.html"
fig.write_html(html_output_path)

fig.show()
print(f"ETABS 3D Model preview saved to ISO 19650 container: {html_output_path}")

Rendering ETABS 3D Structural Model Visualization...



ETABS 3D Model preview saved to ISO 19650 container: 02_SHARED/ETABS_3D_Structural_Model_Preview.html


In [56]:
import os
import requests

# Configuration
ACCESS_TOKEN = os.getenv("LINKEDIN_ACCESS_TOKEN")  # Or paste your token directly
API_VERSION_HEADER = {"X-Restli-Protocol-Version": "2.0.0"}


def get_person_urn(access_token):
  """Fetches the authenticated user's LinkedIn Person URN."""
  url = "https://api.linkedin.com/v2/userinfo"
  headers = {
      "Authorization": f"Bearer {access_token}",
      **API_VERSION_HEADER,
  }
  response = requests.get(url, headers=headers)
  response.raise_for_status()
  data = response.json()
  # Userinfo returns 'sub' as the ID
  return f"urn:li:person:{data['sub']}"


def create_text_post(access_token, author_urn, text_content, article_url=None):
  """Publishes a text or article-sharing post to LinkedIn."""
  url = "https://api.linkedin.com/v2/ugcPosts"
  headers = {
      "Authorization": f"Bearer {access_token}",
      "Content-Type": "application/json",
      **API_VERSION_HEADER,
  }

  # Construct share content structure
  specific_content = {
      "com.linkedin.ugc.ShareContent": {
          "shareCommentary": {"text": text_content},
          "shareMediaCategory": "NONE",
      }
  }

  if article_url:
    specific_content["com.linkedin.ugc.ShareContent"][
        "shareMediaCategory"
    ] = "ARTICLE"
    specific_content["com.linkedin.ugc.ShareContent"]["media"] = [{
        "status": "READY",
        "originalUrl": article_url,
        "title": {"text": "Project Repository & Technical Documentation"},
    }]

  payload = {
      "author": author_urn,
      "lifecycleState": "PUBLISHED",
      "specificContent": specific_content,
      "visibility": {"com.linkedin.ugc.MemberNetworkVisibility": "PUBLIC"},
  }

  response = requests.post(url, headers=headers, json=payload)
  response.raise_for_status()
  return response.json()


# Execution Workflow
if __name__ == "__main__":
  try:
    print("Fetching LinkedIn profile URN...")
    author_urn = get_person_urn(ACCESS_TOKEN)
    print(f"Author URN resolved: {author_urn}")

    post_text = (
        "Excited to share the latest milestone on our digital construction"
        " workflow automation! Automated structural detailing and layout"
        " generation scripts are now live. Check out the repository for"
        " implementation details. #BIM #StructuralEngineering #Python"
        " #Automation"
    )

    project_repo_url = "https://github.com/your-username/your-repo"

    print("Publishing post to LinkedIn...")
    result = create_text_post(
        ACCESS_TOKEN, author_urn, post_text, article_url=project_repo_url
    )
    print("Post published successfully!")
    print(result)

  except requests.exceptions.HTTPError as err:
    print(f"API Error: {err.response.status_code} - {err.response.text}")

Fetching LinkedIn profile URN...
API Error: 401 - {"status":401,"serviceErrorCode":65600,"code":"INVALID_ACCESS_TOKEN","message":"Invalid access token"}


In [58]:
# Step 1: Install required dependencies
!pip install -q reportlab

import os
from google.colab import files
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.lib.units import inch
from reportlab.graphics.shapes import Drawing, Line, Rect, String
from reportlab.platypus import (
    HRFlowable,
    KeepTogether,
    PageBreak,
    Paragraph,
    SimpleDocTemplate,
    Spacer,
    Table,
    TableStyle,
)


def build_rebar_pdf(filename="Structural_Rebar_Detailing_Report.pdf"):
  # Document Setup
  doc = SimpleDocTemplate(
      filename,
      pagesize=A4,
      leftMargin=36,
      rightMargin=36,
      topMargin=36,
      bottomMargin=36,
  )

  story = []
  styles = getSampleStyleSheet()

  # Custom Styles
  title_style = ParagraphStyle(
      "DocTitle",
      parent=styles["Heading1"],
      fontName="Helvetica-Bold",
      fontSize=20,
      leading=24,
      textColor=colors.HexColor("#1A365D"),
  )
  section_style = ParagraphStyle(
      "SectionHeader",
      parent=styles["Heading2"],
      fontName="Helvetica-Bold",
      fontSize=13,
      leading=16,
      textColor=colors.HexColor("#1A365D"),
      spaceBefore=10,
      spaceAfter=6,
  )
  body_style = ParagraphStyle(
      "Body",
      parent=styles["BodyText"],
      fontName="Helvetica",
      fontSize=9,
      leading=12,
      textColor=colors.HexColor("#2D3748"),
  )
  th_style = ParagraphStyle(
      "TableHeader",
      parent=styles["Normal"],
      fontName="Helvetica-Bold",
      fontSize=8,
      leading=10,
      textColor=colors.white,
      alignment=1,
  )
  td_style = ParagraphStyle(
      "TableCell",
      parent=styles["Normal"],
      fontName="Helvetica",
      fontSize=8,
      leading=10,
      alignment=1,
  )

  # 1. Document Header & Title Block
  story.append(
      Paragraph(
          "STRUCTURAL REBAR DETAILING & BAR BENDING SCHEDULE (BBS)", title_style
      )
  )
  story.append(
      Paragraph(
          "Complete Engineering Reinforcement Report | Entire Superstructure",
          body_style,
      )
  )
  story.append(Spacer(1, 8))
  story.append(
      HRFlowable(
          width="100%",
          thickness=2,
          color=colors.HexColor("#1A365D"),
          spaceAfter=15,
      )
  )

  # Project Info Block
  meta_data = [
      [
          Paragraph("<b>Project:</b> Multi-Storey RC Building", body_style),
          Paragraph("<b>Design Code:</b> IS 456 / IS 13920", body_style),
      ],
      [
          Paragraph("<b>Concrete Grade:</b> M30", body_style),
          Paragraph("<b>Steel Grade:</b> Fe500D (TMT)", body_style),
      ],
      [
          Paragraph("<b>Clear Cover:</b> Footing 50mm | Col 40mm | Beam 25mm", body_style),
          Paragraph("<b>Status:</b> Approved for Construction", body_style),
      ],
  ]
  meta_table = Table(meta_data, colWidths=[260, 260])
  meta_table.setStyle(
      TableStyle([
          ("BACKGROUND", (0, 0), (-1, -1), colors.HexColor("#F7FAFC")),
          ("BOX", (0, 0), (-1, -1), 1, colors.HexColor("#CBD5E0")),
          ("INNERGRID", (0, 0), (-1, -1), 0.5, colors.HexColor("#E2E8F0")),
          ("TOPPADDING", (0, 0), (-1, -1), 6),
          ("BOTTOMPADDING", (0, 0), (-1, -1), 6),
          ("LEFTPADDING", (0, 0), (-1, -1), 8),
      ])
  )
  story.append(meta_table)
  story.append(Spacer(1, 15))

  # 2. Section: Column Reinforcement Details
  story.append(Paragraph("1. Column Detailing (C1 - 450mm x 450mm)", section_style))

  # Vector Drawing: Column Cross-Section
  d_col = Drawing(520, 140)
  d_col.add(
      Rect(
          180,
          10,
          120,
          120,
          fillColor=colors.HexColor("#EDF2F7"),
          strokeColor=colors.HexColor("#2D3748"),
          strokeWidth=2,
      )
  )  # Concrete boundary
  d_col.add(
      Rect(
          190,
          20,
          100,
          100,
          fillColor=None,
          strokeColor=colors.HexColor("#E53E3E"),
          strokeWidth=1.5,
      )
  )  # Stirrup/Tie

  # Rebar dots (8 Nos - 20mm Dia)
  rebars = [
      (193, 23),
      (240, 23),
      (287, 23),
      (193, 70),
      (287, 70),
      (193, 117),
      (240, 117),
      (287, 117),
  ]
  for rx, ry in rebars:
    d_col.add(
        Rect(
            rx - 3,
            ry - 3,
            6,
            6,
            fillColor=colors.HexColor("#1A365D"),
            strokeColor=colors.HexColor("#1A365D"),
        )
    )

  d_col.add(
      String(
          185,
          123,
          "8-T20 Main Bars",
          fontSize=8,
          fontName="Helvetica-Bold",
          fillColor=colors.HexColor("#1A365D"),
      )
  )
  d_col.add(
      String(
          300,
          65,
          "T8 @ 150mm c/c Ties",
          fontSize=8,
          fontName="Helvetica-Bold",
          fillColor=colors.HexColor("#E53E3E"),
      )
  )
  story.append(d_col)
  story.append(Spacer(1, 10))

  # 3. Bar Bending Schedule (BBS) Table
  story.append(
      Paragraph("2. Master Bar Bending Schedule (BBS)", section_style)
  )

  bbs_headers = [
      Paragraph("<b>Member ID</b>", th_style),
      Paragraph("<b>Bar Type</b>", th_style),
      Paragraph("<b>Dia (mm)</b>", th_style),
      Paragraph("<b>No. of Bars</b>", th_style),
      Paragraph("<b>Cutting Length (m)</b>", th_style),
      Paragraph("<b>Total Length (m)</b>", th_style),
      Paragraph("<b>Weight (kg)</b>", th_style),
  ]

  # Raw BBS Data Calculation: [Member, Bar Description, Dia (mm), Count, Cut_Length (m)]
  raw_bbs = [
      ["Footing (F1)", "Bottom Mesh (Both Ways)", 12, 24, 2.30],
      ["Column (C1)", "Main Longitudinal Rebar", 20, 8, 4.50],
      ["Column (C1)", "Rectangular Ties / Links", 8, 30, 1.55],
      ["Beam (B1)", "Top Continuous Rebar", 16, 4, 6.20],
      ["Beam (B1)", "Bottom Continuous Rebar", 16, 3, 6.20],
      ["Beam (B1)", "Shear Stirrups", 8, 42, 1.25],
      ["Slab (S1)", "Main Reinforcement", 10, 35, 3.80],
      ["Slab (S1)", "Distribution Steel", 8, 25, 4.20],
  ]

  bbs_rows = [bbs_headers]
  grand_total_weight = 0.0

  for member, bar_desc, dia, count, cut_len in raw_bbs:
    tot_len = count * cut_len
    # Weight per meter = (D^2 / 162) kg/m
    unit_wt = (dia**2) / 162.0
    weight = tot_len * unit_wt
    grand_total_weight += weight

    bbs_rows.append([
        Paragraph(member, td_style),
        Paragraph(bar_desc, td_style),
        Paragraph(str(dia), td_style),
        Paragraph(str(count), td_style),
        Paragraph(f"{cut_len:.2f}", td_style),
        Paragraph(f"{tot_len:.2f}", td_style),
        Paragraph(f"{weight:.2f}", td_style),
    ])

  # Total Summary Row
  bbs_rows.append([
      Paragraph("<b>GRAND TOTAL STEEL WEIGHT</b>", td_style),
      "",
      "",
      "",
      "",
      "",
      Paragraph(f"<b>{grand_total_weight:.2f} kg</b>", td_style),
  ])

  bbs_table = Table(
      bbs_rows, colWidths=[90, 130, 50, 60, 70, 60, 60], repeatRows=1
  )
  bbs_table.setStyle(
      TableStyle([
          ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#1A365D")),
          ("ALIGN", (0, 0), (-1, -1), "CENTER"),
          ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
          ("GRID", (0, 0), (-1, -1), 0.5, colors.HexColor("#CBD5E0")),
          ("BACKGROUND", (0, -1), (-1, -1), colors.HexColor("#EDF2F7")),
          ("SPAN", (0, -1), (5, -1)),
          ("TOPPADDING", (0, 0), (-1, -1), 5),
          ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
      ])
  )
  story.append(bbs_table)
  story.append(Spacer(1, 15))

  # 4. Section: Beam Longitudinal Detailing Drawing
  story.append(
      Paragraph("3. Typical Beam Rebar Profile (B1 - 300x500)", section_style)
  )

  d_beam = Drawing(520, 100)
  # Concrete Beam Outline
  d_beam.add(
      Rect(
          20,
          20,
          480,
          60,
          fillColor=colors.HexColor("#F7FAFC"),
          strokeColor=colors.HexColor("#2D3748"),
          strokeWidth=1.5,
      )
  )
  # Top Rebar
  d_beam.add(
      Line(
          25,
          70,
          495,
          70,
          strokeColor=colors.HexColor("#E53E3E"),
          strokeWidth=2,
      )
  )
  # Bottom Rebar
  d_beam.add(
      Line(
          25,
          30,
          495,
          30,
          strokeColor=colors.HexColor("#1A365D"),
          strokeWidth=2,
      )
  )

  # Stirrup vertical lines across span
  for sx in range(40, 490, 30):
    d_beam.add(
        Line(
            sx,
            30,
            sx,
            70,
            strokeColor=colors.HexColor("#A0AEC0"),
            strokeWidth=1,
        )
    )

  d_beam.add(
      String(
          30,
          75,
          "Top Rebar: 4-T16 Continuous",
          fontSize=8,
          fontName="Helvetica-Bold",
          fillColor=colors.HexColor("#E53E3E"),
      )
  )
  d_beam.add(
      String(
          30,
          15,
          "Bottom Rebar: 3-T16 Continuous",
          fontSize=8,
          fontName="Helvetica-Bold",
          fillColor=colors.HexColor("#1A365D"),
      )
  )
  d_beam.add(
      String(
          360,
          45,
          "2-Legged T8 @ 125mm c/c",
          fontSize=8,
          fontName="Helvetica-Bold",
          fillColor=colors.HexColor("#4A5568"),
      )
  )
  story.append(d_beam)

  # Build Document
  doc.build(story)
  print(f"PDF successfully generated: {filename}")


# Execute generation and trigger download in Google Colab
if __name__ == "__main__":
  output_filename = "Structural_Rebar_Detailing_Report.pdf"
  build_rebar_pdf(output_filename)
  files.download(output_filename)

PDF successfully generated: Structural_Rebar_Detailing_Report.pdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>